# 02 – Bayesian Beta Regression | Garment Worker Productivity

This notebook builds a **hierarchical Beta regression** using **PyMC v5** to model garment worker productivity (a continuous outcome bounded in (0, 1)).

Key features:
- **Three nested models** (A → B → C) of increasing complexity, compared via PSIS-LOO
- **Monotone I-splines** to enforce a non-decreasing incentive effect
- **Interaction terms** (incentive×WIP, WIP×workers) in the full model
- **Partial pooling** of team-level intercepts via non-centered parameterization
- **Comprehensive metrics** per model: MSE, RMSE, MAE, R², AUC-ROC, Accuracy (threshold=0.75)
- **Side-by-side model comparison** table and bar chart

> **Prerequisite:** `01_data_prep.ipynb` must have been run to produce `garments_clean.csv`, `data_splits.pkl`, and the `outputs/` directory.

## 1. Colab Setup

In [ ]:
# ── Colab setup ──────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymc", "arviz"], check=True)
print("Done installing")

In [ ]:
from google.colab import files
import shutil
uploaded = files.upload()  # select data_splits.pkl
shutil.move("data_splits.pkl", "/content/data_splits.pkl")

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pickle
import warnings
from pathlib import Path

import pymc as pm
import pytensor.tensor as pt
import arviz as az
from scipy.special import expit, logit
from scipy.interpolate import BSpline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, accuracy_score
)

warnings.filterwarnings('ignore')
DATA_DIR  = Path("/content")
OUT_DIR   = DATA_DIR / "outputs"
MODEL_DIR = DATA_DIR / "models"
OUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style="whitegrid")
print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")

## 3. Load Processed Data

In [ ]:
# Upgrade pandas to resolve potential TypeError with StringDtype
!pip install pandas --upgrade -q

In [ ]:
# Restart runtime to apply pandas upgrade
import os
os.kill(os.getpid(), 9)

### ⚠️ Please run the above two cells, wait for the runtime to restart, and then run the following cell.

In [ ]:
import pickle
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content")

with open(DATA_DIR / "data_splits.pkl", "rb") as f:
    splits = pickle.load(f)

df_train = splits['df_train'].copy()
df_val   = splits['df_val'].copy()
df_test  = splits['df_test'].copy()

# Combine train+val for model fitting
df_fit = pd.concat([df_train, df_val], ignore_index=True).sort_values('date')

print(f"Fit set:  {len(df_fit)} rows")
print(f"Test set: {len(df_test)} rows")
print(df_fit[['actual_productivity', 'incentive', 'wip', 'over_time', 'no_of_workers']].describe())

## 4. Feature Preparation

In [ ]:
# Clip actual_productivity to strict (0,1) – Beta distribution requires open interval
EPS = 1e-4
y_fit  = np.clip(df_fit['actual_productivity'].values,  EPS, 1 - EPS)
y_test = np.clip(df_test['actual_productivity'].values, EPS, 1 - EPS)

# Standardize continuous features (fit on df_fit only)
feat_cols = ['incentive', 'wip', 'over_time', 'smv', 'no_of_workers',
             'no_of_style_change', 'targeted_productivity']

scaler2 = StandardScaler()
X_fit_raw  = df_fit[feat_cols].fillna(0).values
X_test_raw = df_test[feat_cols].fillna(0).values
X_fit  = scaler2.fit_transform(X_fit_raw)
X_test = scaler2.transform(X_test_raw)

# Team indices (0-based integer)
teams = sorted(df_fit['team'].unique())
team_map = {t: i for i, t in enumerate(teams)}
team_idx_fit  = df_fit['team'].map(team_map).values.astype(int)
team_idx_test = df_test['team'].map(team_map).fillna(0).values.astype(int)
n_teams = len(teams)

# Department and WIP-missing dummies
dept_fit  = (df_fit['department']  == 'sewing').astype(float).values
dept_test = (df_test['department'] == 'sewing').astype(float).values
wip_missing_fit  = df_fit['wip_missing'].values.astype(float)  if 'wip_missing' in df_fit.columns  else np.zeros(len(df_fit))
wip_missing_test = df_test['wip_missing'].values.astype(float) if 'wip_missing' in df_test.columns else np.zeros(len(df_test))

COL = {name: i for i, name in enumerate(feat_cols)}
print("Feature columns:", feat_cols)
print(f"Teams: {n_teams}, index range: {team_idx_fit.min()}–{team_idx_fit.max()}")

## 5. Helper Functions

### 5a. I-Spline Basis (Monotone Incentive Effect)

In [ ]:
def b_spline_basis(x, knots, degree=3):
    """B-spline basis matrix via scipy."""
    t = np.concatenate([np.repeat(knots[0], degree), knots, np.repeat(knots[-1], degree)])
    n_basis = len(t) - degree - 1
    B = np.zeros((len(x), n_basis))
    for i in range(n_basis):
        c = np.zeros(n_basis); c[i] = 1.0
        spl = BSpline(t, c, degree)
        B[:, i] = spl(x)
    return B

def i_spline_basis(x, knots, degree=3):
    """Integrated B-splines → monotone increasing with non-negative weights."""
    B = b_spline_basis(x, knots, degree)
    I = np.cumsum(B[:, ::-1], axis=1)[:, ::-1]
    return I[:, :-1]

# Build I-spline basis on standardized incentive values
incentive_std_fit  = X_fit[:, COL['incentive']]
incentive_std_test = X_test[:, COL['incentive']]
knots_inc = np.quantile(incentive_std_fit, np.linspace(0, 1, 8))
I_fit  = i_spline_basis(incentive_std_fit,  knots_inc).astype(np.float32)
I_test = i_spline_basis(incentive_std_test, knots_inc).astype(np.float32)
n_spline = I_fit.shape[1]
print(f"I-spline basis shape: {I_fit.shape}")

### 5b. Metrics Computation Function

Computes **MSE, RMSE, MAE, R², AUC-ROC, Accuracy** for a given model's predictions.

In [ ]:
PRODUCTIVITY_THRESHOLD = 0.75  # workers above this are classified as "high productivity"

def compute_metrics(y_true, y_pred_mean, model_name="Model"):
    """
    Compute regression + classification metrics.

    Parameters
    ----------
    y_true       : array of true productivity values (0–1)
    y_pred_mean  : array of posterior-mean predictions (0–1)
    model_name   : string label

    Returns
    -------
    dict with keys: Model, MSE, RMSE, MAE, R2, AUC, Accuracy
    """
    mse  = mean_squared_error(y_true, y_pred_mean)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred_mean)
    r2   = r2_score(y_true, y_pred_mean)

    # Binary labels: high productivity = above threshold
    y_true_bin = (y_true >= PRODUCTIVITY_THRESHOLD).astype(int)
    y_pred_bin = (y_pred_mean >= PRODUCTIVITY_THRESHOLD).astype(int)

    # AUC needs probability scores; use the continuous prediction as score
    try:
        auc = roc_auc_score(y_true_bin, y_pred_mean)
    except ValueError:
        auc = float('nan')  # only one class present

    acc = accuracy_score(y_true_bin, y_pred_bin)

    metrics = {
        "Model":    model_name,
        "MSE":      round(mse,  5),
        "RMSE":     round(rmse, 5),
        "MAE":      round(mae,  5),
        "R²":       round(r2,   5),
        "AUC-ROC":  round(auc,  5),
        "Accuracy": round(acc,  5),
    }
    return metrics

# Store all model metrics here
all_metrics = []
print(f"Metrics ready. Classification threshold = {PRODUCTIVITY_THRESHOLD}")

## 6. Model A – Baseline Beta Regression

Simple global Beta regression with linear effects, no team structure or splines.

In [ ]:
coords_A = {"obs": np.arange(len(y_fit))}

with pm.Model(coords=coords_A) as model_A:
    incentive_data = pm.Data("incentive", X_fit[:, COL['incentive']], dims="obs")
    wip_data       = pm.Data("wip",       X_fit[:, COL['wip']],       dims="obs")
    ot_data        = pm.Data("over_time", X_fit[:, COL['over_time']], dims="obs")
    smv_data       = pm.Data("smv",       X_fit[:, COL['smv']],       dims="obs")
    nw_data        = pm.Data("nw",        X_fit[:, COL['no_of_workers']], dims="obs")
    dept_data      = pm.Data("dept",      dept_fit,                    dims="obs")
    wip_miss_data  = pm.Data("wip_miss",  wip_missing_fit,             dims="obs")

    intercept   = pm.Normal("intercept",   mu=0, sigma=2)
    b_incentive = pm.Normal("b_incentive", mu=0, sigma=1)
    b_wip       = pm.Normal("b_wip",       mu=0, sigma=1)
    b_ot        = pm.Normal("b_ot",        mu=0, sigma=1)
    b_smv       = pm.Normal("b_smv",       mu=0, sigma=1)
    b_nw        = pm.Normal("b_nw",        mu=0, sigma=1)
    b_dept      = pm.Normal("b_dept",      mu=0, sigma=1)
    b_wip_miss  = pm.Normal("b_wip_miss",  mu=0, sigma=1)
    nu          = pm.Exponential("nu",     lam=0.1)

    eta = (intercept
           + b_incentive * incentive_data
           + b_wip       * wip_data
           + b_ot        * ot_data
           + b_smv       * smv_data
           + b_nw        * nw_data
           + b_dept      * dept_data
           + b_wip_miss  * wip_miss_data)

    mu_A  = pm.Deterministic("mu_A", pm.math.invlogit(eta), dims="obs")
    y_obs = pm.Beta("y_obs", mu=mu_A, nu=nu, observed=y_fit, dims="obs")

    idata_A = pm.sample(500, tune=500, chains=2, target_accept=0.9,
                        idata_kwargs={"log_likelihood": True},
                        random_seed=42, progressbar=True)

print("\nModel A — Baseline Beta Regression")
print(az.summary(idata_A, var_names=["intercept","b_incentive","b_wip","b_ot","b_smv","b_nw","b_dept","b_wip_miss","nu"]))

In [ ]:
# ── Model A: Test-set predictions & metrics ─────────────────────────────
coords_test_A = {"obs": np.arange(len(y_test))}

with model_A:
    pm.set_data({
        "incentive": X_test[:, COL['incentive']],
        "wip":       X_test[:, COL['wip']],
        "over_time": X_test[:, COL['over_time']],
        "smv":       X_test[:, COL['smv']],
        "nw":        X_test[:, COL['no_of_workers']],
        "dept":      dept_test,
        "wip_miss":  wip_missing_test,
    }, coords=coords_test_A)
    ppc_A = pm.sample_posterior_predictive(
        idata_A, random_seed=42, var_names=["mu_A"], return_inferencedata=False
    )

mu_A_samples = ppc_A["mu_A"].reshape(-1, len(y_test))
mu_A_mean    = mu_A_samples.mean(axis=0)

metrics_A = compute_metrics(y_test, mu_A_mean, "Model A (Baseline)")
all_metrics.append(metrics_A)
print("Model A — Test Metrics")
for k, v in metrics_A.items():
    print(f"  {k:<12}: {v}")

## 7. Model B – Hierarchical Beta Regression with Team Random Effects

In [ ]:
team_coords = [f"team_{t}" for t in teams]
coords_B = {"obs": np.arange(len(y_fit)), "team": team_coords}

with pm.Model(coords=coords_B) as model_B:
    incentive_data = pm.Data("incentive", X_fit[:, COL['incentive']], dims="obs")
    wip_data       = pm.Data("wip",       X_fit[:, COL['wip']],       dims="obs")
    ot_data        = pm.Data("over_time", X_fit[:, COL['over_time']], dims="obs")
    smv_data       = pm.Data("smv",       X_fit[:, COL['smv']],       dims="obs")
    nw_data        = pm.Data("nw",        X_fit[:, COL['no_of_workers']], dims="obs")
    dept_data      = pm.Data("dept",      dept_fit,                    dims="obs")
    wip_miss_data  = pm.Data("wip_miss",  wip_missing_fit,             dims="obs")
    team_data      = pm.Data("team_idx",  team_idx_fit,                dims="obs")

    b_incentive = pm.Normal("b_incentive", mu=0, sigma=1)
    b_wip       = pm.Normal("b_wip",       mu=0, sigma=1)
    b_ot        = pm.Normal("b_ot",        mu=0, sigma=1)
    b_smv       = pm.Normal("b_smv",       mu=0, sigma=1)
    b_nw        = pm.Normal("b_nw",        mu=0, sigma=1)
    b_dept      = pm.Normal("b_dept",      mu=0, sigma=1)
    b_wip_miss  = pm.Normal("b_wip_miss",  mu=0, sigma=1)

    # Hierarchical team intercepts (non-centered)
    mu_team    = pm.Normal("mu_team",    mu=0, sigma=2)
    sigma_team = pm.Exponential("sigma_team", lam=1)
    z_team     = pm.Normal("z_team", mu=0, sigma=1, dims="team")
    alpha_team = pm.Deterministic("alpha_team", mu_team + z_team * sigma_team, dims="team")
    nu = pm.Exponential("nu", lam=0.1)

    eta = (alpha_team[team_data]
           + b_incentive * incentive_data
           + b_wip       * wip_data
           + b_ot        * ot_data
           + b_smv       * smv_data
           + b_nw        * nw_data
           + b_dept      * dept_data
           + b_wip_miss  * wip_miss_data)

    mu_B = pm.Deterministic("mu_B", pm.math.invlogit(eta), dims="obs")
    pm.Beta("y_obs", mu=mu_B, nu=nu, observed=y_fit, dims="obs")

    idata_B = pm.sample(1000, tune=1000, chains=4, target_accept=0.9,
                        idata_kwargs={"log_likelihood": True},
                        random_seed=42, progressbar=True)

print("\nModel B — Hierarchical Team Effects")
print(az.summary(idata_B, var_names=["mu_team","sigma_team","b_incentive","b_wip","b_ot","nu"]))

In [ ]:
# ── Model B: Test-set predictions & metrics ─────────────────────────────
coords_test_B = {"obs": np.arange(len(y_test))}

with model_B:
    pm.set_data({
        "incentive": X_test[:, COL['incentive']],
        "wip":       X_test[:, COL['wip']],
        "over_time": X_test[:, COL['over_time']],
        "smv":       X_test[:, COL['smv']],
        "nw":        X_test[:, COL['no_of_workers']],
        "dept":      dept_test,
        "wip_miss":  wip_missing_test,
        "team_idx":  team_idx_test,
    }, coords=coords_test_B)
    ppc_B = pm.sample_posterior_predictive(
        idata_B, random_seed=42, var_names=["mu_B"], return_inferencedata=False
    )

mu_B_samples = ppc_B["mu_B"].reshape(-1, len(y_test))
mu_B_mean    = mu_B_samples.mean(axis=0)

metrics_B = compute_metrics(y_test, mu_B_mean, "Model B (Hierarchical)")
all_metrics.append(metrics_B)
print("Model B — Test Metrics")
for k, v in metrics_B.items():
    print(f"  {k:<12}: {v}")

## 8. Model C – Hierarchical + Monotone Spline on Incentive + Interactions

In [ ]:
inc_x_wip = (X_fit[:, COL['incentive']] * X_fit[:, COL['wip']])
wip_x_nw  = (X_fit[:, COL['wip']]       * X_fit[:, COL['no_of_workers']])

coords_C = {"obs": np.arange(len(y_fit)), "team": team_coords, "spline": np.arange(n_spline)}

with pm.Model(coords=coords_C) as model_C:
    I_data     = pm.Data("I_spline",  I_fit,                        dims=("obs","spline"))
    wip_data   = pm.Data("wip",       X_fit[:, COL['wip']],          dims="obs")
    ot_data    = pm.Data("over_time", X_fit[:, COL['over_time']],    dims="obs")
    smv_data   = pm.Data("smv",       X_fit[:, COL['smv']],          dims="obs")
    nw_data    = pm.Data("nw",        X_fit[:, COL['no_of_workers']], dims="obs")
    dept_data  = pm.Data("dept",      dept_fit,                       dims="obs")
    wip_miss_d = pm.Data("wip_miss",  wip_missing_fit,                dims="obs")
    team_data  = pm.Data("team_idx",  team_idx_fit,                   dims="obs")
    ix_inc_wip = pm.Data("ix_inc_wip", inc_x_wip,                    dims="obs")
    ix_wip_nw  = pm.Data("ix_wip_nw",  wip_x_nw,                     dims="obs")

    # Monotone spline weights
    w_spline = pm.HalfNormal("w_spline", sigma=1, dims="spline")
    spline_effect = pm.Deterministic("spline_effect", pt.dot(I_data, w_spline), dims="obs")

    b_wip      = pm.Normal("b_wip",      mu=0, sigma=1)
    b_ot       = pm.Normal("b_ot",       mu=0, sigma=1)
    b_smv      = pm.Normal("b_smv",      mu=0, sigma=1)
    b_nw       = pm.Normal("b_nw",       mu=0, sigma=1)
    b_dept     = pm.Normal("b_dept",     mu=0, sigma=1)
    b_wip_miss = pm.Normal("b_wip_miss", mu=0, sigma=1)
    b_inc_wip  = pm.Normal("b_inc_wip",  mu=0, sigma=0.5)
    b_wip_nw   = pm.Normal("b_wip_nw",   mu=0, sigma=0.5)

    mu_team    = pm.Normal("mu_team",    mu=0, sigma=2)
    sigma_team = pm.Exponential("sigma_team", lam=1)
    z_team     = pm.Normal("z_team", mu=0, sigma=1, dims="team")
    alpha_team = pm.Deterministic("alpha_team", mu_team + z_team * sigma_team, dims="team")
    nu = pm.Exponential("nu", lam=0.1)

    eta = (alpha_team[team_data]
           + spline_effect
           + b_wip      * wip_data
           + b_ot       * ot_data
           + b_smv      * smv_data
           + b_nw       * nw_data
           + b_dept     * dept_data
           + b_wip_miss * wip_miss_d
           + b_inc_wip  * ix_inc_wip
           + b_wip_nw   * ix_wip_nw)

    mu_C = pm.Deterministic("mu_C", pm.math.clip(pm.math.invlogit(eta), EPS, 1 - EPS), dims="obs")
    pm.Beta("y_obs", mu=mu_C, nu=nu, observed=y_fit, dims="obs")

    idata_C = pm.sample(1000, tune=1000, chains=4, target_accept=0.92,
                        idata_kwargs={"log_likelihood": True},
                        random_seed=42, progressbar=True)

print("\nModel C — Hierarchical + I-Spline + Interactions")
print(az.summary(idata_C, var_names=["mu_team","sigma_team","w_spline","b_wip","b_ot","b_inc_wip","b_wip_nw","nu"]))

In [ ]:
# ── Model C: Test-set predictions & metrics ─────────────────────────────
inc_x_wip_test = X_test[:, COL['incentive']] * X_test[:, COL['wip']]
wip_x_nw_test  = X_test[:, COL['wip']]       * X_test[:, COL['no_of_workers']]
coords_test_C  = {"obs": np.arange(len(y_test))}

with model_C:
    pm.set_data({
        "I_spline":  I_test,
        "wip":       X_test[:, COL['wip']],
        "over_time": X_test[:, COL['over_time']],
        "smv":       X_test[:, COL['smv']],
        "nw":        X_test[:, COL['no_of_workers']],
        "dept":      dept_test,
        "wip_miss":  wip_missing_test,
        "team_idx":  team_idx_test,
        "ix_inc_wip": inc_x_wip_test,
        "ix_wip_nw":  wip_x_nw_test,
    }, coords=coords_test_C)
    ppc_C = pm.sample_posterior_predictive(
        idata_C, random_seed=42, var_names=["mu_C", "y_obs"], return_inferencedata=False
    )

mu_C_samples = ppc_C["mu_C"].reshape(-1, len(y_test))
mu_C_mean    = mu_C_samples.mean(axis=0)

metrics_C = compute_metrics(y_test, mu_C_mean, "Model C (Spline+Interact)")
all_metrics.append(metrics_C)
print("Model C — Test Metrics")
for k, v in metrics_C.items():
    print(f"  {k:<12}: {v}")

## 9. PSIS-LOO Model Comparison

In [ ]:
comparison = az.compare(
    {"Model_A_baseline": idata_A,
     "Model_B_hierarchical": idata_B,
     "Model_C_spline_interaction": idata_C},
    ic="loo", method="stacking", scale="log"
)
print("\nPSIS-LOO Model Comparison:")
print(comparison)

fig, ax = plt.subplots(figsize=(8, 4))
az.plot_compare(comparison, insample_dev=False, ax=ax)
ax.set_title("PSIS-LOO Model Comparison")
plt.tight_layout()
plt.savefig(OUT_DIR / "02_model_comparison_loo.png", dpi=120)
plt.show()

## 10. 📊 Full Metrics Comparison Table & Plots

Side-by-side comparison of **MSE, RMSE, MAE, R², AUC-ROC, Accuracy** across all three models on the held-out test set.

In [ ]:
# ─── Build comparison DataFrame ──────────────────────────────────────────
metrics_df = pd.DataFrame(all_metrics).set_index("Model")

# Pretty-print with colour highlighting
print("\n" + "="*75)
print(" MODEL COMPARISON — Test-Set Metrics")
print(f" Classification threshold: {PRODUCTIVITY_THRESHOLD}")
print("="*75)
print(metrics_df.to_string())
print("="*75)

# Show best value per metric
print("\n✅ Best value per metric:")
best = {}
for col in metrics_df.columns:
    if col in ["MSE", "RMSE", "MAE"]:
        best_idx = metrics_df[col].idxmin()
        best[col] = (best_idx, metrics_df.loc[best_idx, col])
        print(f"  {col:<12}: {best_idx}  ({metrics_df.loc[best_idx, col]})  ↓ lower is better")
    else:
        best_idx = metrics_df[col].idxmax()
        best[col] = (best_idx, metrics_df.loc[best_idx, col])
        print(f"  {col:<12}: {best_idx}  ({metrics_df.loc[best_idx, col]})  ↑ higher is better")

In [ ]:
# ─── Bar chart comparison for all metrics ────────────────────────────────
metric_cols  = ["MSE", "RMSE", "MAE", "R²", "AUC-ROC", "Accuracy"]
model_labels = metrics_df.index.tolist()
n_metrics    = len(metric_cols)
x            = np.arange(len(model_labels))

colors = ["#4C72B0", "#DD8452", "#55A868"]  # one per model

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for ax, metric in zip(axes, metric_cols):
    vals = metrics_df[metric].values
    bars = ax.bar(x, vals, color=colors, edgecolor="white", width=0.6)

    # Annotate bars
    for bar, val in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(vals) * 0.01,
            f"{val:.4f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold"
        )

    # Highlight best bar
    if metric in ["MSE", "RMSE", "MAE"]:
        best_i = np.argmin(vals)
        direction = "↓ lower is better"
    else:
        best_i = np.argmax(vals)
        direction = "↑ higher is better"

    bars[best_i].set_edgecolor("black")
    bars[best_i].set_linewidth(2.5)

    ax.set_title(f"{metric}  ({direction})", fontsize=11, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(
        [m.replace(" (", "\n(") for m in model_labels],
        fontsize=8
    )
    ax.set_ylabel(metric)
    ax.grid(axis="y", alpha=0.4)
    sns.despine(ax=ax)

fig.suptitle(
    f"Model Comparison — Test-Set Performance Metrics\n"
    f"(Classification threshold = {PRODUCTIVITY_THRESHOLD}; bold border = best)",
    fontsize=13, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig(OUT_DIR / "02_metrics_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: outputs/02_metrics_comparison.png")

In [ ]:
# ─── Radar chart for visual overview ────────────────────────────────────
import matplotlib.patches as mpatches

# Normalise metrics so 1.0 = best, 0.0 = worst (for radar readability)
radar_metrics = ["RMSE", "MAE", "R²", "AUC-ROC", "Accuracy"]
radar_df = metrics_df[radar_metrics].copy()

radar_norm = pd.DataFrame(index=radar_df.index)
for col in radar_metrics:
    col_vals = radar_df[col]
    if col in ["RMSE", "MAE"]:  # lower is better → invert
        radar_norm[col] = 1 - (col_vals - col_vals.min()) / (col_vals.max() - col_vals.min() + 1e-12)
    else:                        # higher is better
        radar_norm[col] = (col_vals - col_vals.min()) / (col_vals.max() - col_vals.min() + 1e-12)

categories = radar_metrics
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=11)
ax.set_ylim(0, 1)

for (model_name, row), color in zip(radar_norm.iterrows(), colors):
    values = row.tolist() + row.tolist()[:1]
    ax.plot(angles, values, color=color, linewidth=2, label=model_name)
    ax.fill(angles, values, color=color, alpha=0.15)

ax.set_title("Model Performance Radar\n(normalised; outer edge = best)",
             fontsize=12, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR / "02_metrics_radar.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: outputs/02_metrics_radar.png")

## 11. Posterior Predictive Checks (Model C)

In [ ]:
# Re-attach fit-set data before PPC
coords_fit_C = {"obs": np.arange(len(y_fit))}
with model_C:
    pm.set_data({
        "I_spline":  I_fit,
        "wip":       X_fit[:, COL['wip']],
        "over_time": X_fit[:, COL['over_time']],
        "smv":       X_fit[:, COL['smv']],
        "nw":        X_fit[:, COL['no_of_workers']],
        "dept":      dept_fit,
        "wip_miss":  wip_missing_fit,
        "team_idx":  team_idx_fit,
        "ix_inc_wip": inc_x_wip,
        "ix_wip_nw":  wip_x_nw,
    }, coords=coords_fit_C)
    ppc_C_fit = pm.sample_posterior_predictive(idata_C, random_seed=42)

if not hasattr(idata_C, 'posterior_predictive'):
    idata_C.add_groups(posterior_predictive=ppc_C_fit.posterior_predictive)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
az.plot_ppc(idata_C, observed_rug=True, ax=axes[0], num_pp_samples=200)
axes[0].set_title("Posterior Predictive Check — Model C")
az.plot_loo_pit(idata=idata_C, y="y_obs", ax=axes[1])
axes[1].set_title("LOO-PIT — Model C (uniform = well-calibrated)")
plt.tight_layout()
plt.savefig(OUT_DIR / "02_ppc_and_loopit.png", dpi=120)
plt.show()

## 12. Predicted vs Actual Scatter (All Models)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

pred_dict = {
    "Model A (Baseline)": mu_A_mean,
    "Model B (Hierarchical)": mu_B_mean,
    "Model C (Spline+Interact)": mu_C_mean,
}

for ax, (name, preds), color in zip(axes, pred_dict.items(), colors):
    m = compute_metrics(y_test, preds, name)
    ax.scatter(y_test, preds, alpha=0.35, s=18, color=color)
    ax.plot([0, 1], [0, 1], 'r--', lw=1.5, label="Perfect prediction")
    ax.set_xlabel("Actual Productivity", fontsize=10)
    ax.set_ylabel("Predicted E[μ]", fontsize=10)
    ax.set_title(
        f"{name}\nRMSE={m['RMSE']:.4f}  R²={m['R²']:.4f}\n"
        f"AUC={m['AUC-ROC']:.4f}  Acc={m['Accuracy']:.4f}",
        fontsize=9
    )
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.suptitle("Predicted vs Actual Productivity — All Models (Test Set)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "02_predicted_vs_actual_all.png", dpi=120)
plt.show()
print("Saved: outputs/02_predicted_vs_actual_all.png")

## 13. Residual Distribution (All Models)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

for ax, (name, preds), color in zip(axes, pred_dict.items(), colors):
    residuals = y_test - preds
    ax.hist(residuals, bins=30, color=color, alpha=0.75, edgecolor="white")
    ax.axvline(0, color="black", lw=1.5, ls="--")
    ax.set_title(f"{name}\nMean residual = {residuals.mean():.4f}", fontsize=9)
    ax.set_xlabel("Residual (actual − predicted)", fontsize=9)
    ax.set_ylabel("Count", fontsize=9)
    ax.grid(alpha=0.3)

fig.suptitle("Residual Distributions — All Models (Test Set)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "02_residuals_all.png", dpi=120)
plt.show()

## 14. Posterior Coefficient Plot (Model C)

In [ ]:
az.plot_forest(idata_C,
               var_names=["b_wip","b_ot","b_smv","b_nw","b_dept","b_wip_miss","b_inc_wip","b_wip_nw"],
               combined=True, hdi_prob=0.94, figsize=(8, 5))
plt.title("Posterior Effect Sizes — Model C (94% HDI)")
plt.tight_layout()
plt.savefig(OUT_DIR / "02_posterior_effects.png", dpi=120)
plt.show()

az.plot_forest(idata_C, var_names=["alpha_team"], combined=True,
               hdi_prob=0.94, figsize=(8, 6))
plt.title("Team-Level Intercepts (Partial Pooling) — 94% HDI")
plt.tight_layout()
plt.savefig(OUT_DIR / "02_team_intercepts.png", dpi=120)
plt.show()

## 15. Monotone Spline Visualization (Model C)

In [ ]:
inc_grid_std = np.linspace(incentive_std_fit.min(), incentive_std_fit.max(), 100)
I_grid = i_spline_basis(inc_grid_std, knots_inc).astype(np.float32)
w_samples = idata_C.posterior["w_spline"].values.reshape(-1, n_spline)
spline_grid_samples = w_samples @ I_grid.T
inc_grid_orig = inc_grid_std * scaler2.scale_[COL['incentive']] + scaler2.mean_[COL['incentive']]

fig, ax = plt.subplots(figsize=(8, 4))
az.plot_hdi(inc_grid_orig, spline_grid_samples, hdi_prob=0.80,
            fill_kwargs={"alpha": 0.4, "label": "80% HDI"}, ax=ax)
az.plot_hdi(inc_grid_orig, spline_grid_samples, hdi_prob=0.94,
            fill_kwargs={"alpha": 0.2, "label": "94% HDI"}, ax=ax)
ax.plot(inc_grid_orig, spline_grid_samples.mean(axis=0), color="navy", lw=2, label="Posterior mean")
ax.axvline(69.5, color="red", ls="--", alpha=0.7, label="Paper threshold (69.5 BDT)")
ax.set_xlabel("Incentive (BDT)")
ax.set_ylabel("Effect on logit(productivity)")
ax.set_title("Monotone Spline Effect of Incentive on Productivity (Model C)")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "02_incentive_spline.png", dpi=120)
plt.show()

## 16. Save Posteriors & Metrics

In [ ]:
idata_A.to_netcdf(str(MODEL_DIR / "model_A_baseline.nc"))
idata_B.to_netcdf(str(MODEL_DIR / "model_B_hierarchical.nc"))
idata_C.to_netcdf(str(MODEL_DIR / "model_C_spline_interaction.nc"))

comparison.to_csv(OUT_DIR / "02_loo_comparison.csv")
metrics_df.to_csv(OUT_DIR / "02_test_metrics_all_models.csv")

from google.colab import files
files.download(str(OUT_DIR / "02_loo_comparison.csv"))
files.download(str(OUT_DIR / "02_test_metrics_all_models.csv"))

print("\n" + "="*60)
print(" FINAL SUMMARY")
print("="*60)
print(f"  Best LOO model : {comparison.index[0]}")
print(f"\n  Model A  RMSE={metrics_A['RMSE']:.4f}  AUC={metrics_A['AUC-ROC']:.4f}  Acc={metrics_A['Accuracy']:.4f}")
print(f"  Model B  RMSE={metrics_B['RMSE']:.4f}  AUC={metrics_B['AUC-ROC']:.4f}  Acc={metrics_B['Accuracy']:.4f}")
print(f"  Model C  RMSE={metrics_C['RMSE']:.4f}  AUC={metrics_C['AUC-ROC']:.4f}  Acc={metrics_C['Accuracy']:.4f}")
print("="*60)
print("\nFiles produced:")
print("  models/   model_A_baseline.nc, model_B_hierarchical.nc, model_C_spline_interaction.nc")
print("  outputs/  02_metrics_comparison.png, 02_metrics_radar.png")
print("            02_predicted_vs_actual_all.png, 02_residuals_all.png")
print("            02_model_comparison_loo.png, 02_posterior_effects.png")
print("            02_incentive_spline.png, 02_ppc_and_loopit.png")
print("            02_loo_comparison.csv, 02_test_metrics_all_models.csv")

## 17. Summary & Conclusions

### Model Comparison (PSIS-LOO)

| Model | Structure | Expected LOO rank |
|-------|-----------|-------------------|
| **Model A** | Baseline linear, no team pooling | Weakest |
| **Model B** | + Partial pooling of team intercepts | Middle |
| **Model C** | + Monotone I-spline on incentive + 2 interactions | Best |

### Metrics Interpretation

| Metric | What it measures | Better when |
|--------|-----------------|-------------|
| **MSE** | Mean squared prediction error | Lower |
| **RMSE** | Root MSE (same units as productivity) | Lower |
| **MAE** | Mean absolute error (robust to outliers) | Lower |
| **R²** | Variance explained by the model | Higher (max 1.0) |
| **AUC-ROC** | Ability to rank high vs low producers | Higher (max 1.0) |
| **Accuracy** | % workers correctly classified as high/low productivity | Higher |

Typical test MAE on this dataset is **0.10–0.15**, reflecting genuine irreducible variation in daily productivity. RMSE is slightly higher due to penalising large errors more.

### What Comes Next
- **`03_kalman_dlm.ipynb`** — Dynamic Linear Model for temporal evolution
- **`04_validation.ipynb`** — Full calibration, comparison with frequentist baselines